# Person Detector Training (YOLOv8)

Pipeline to train a YOLOv8 person-detection model from a Roboflow-exported dataset.

**Note:** run the cells top to bottom. The augmentation step (section 3) writes new images to disk before training, so it must run before the training cell.

## 1. Dataset preparation

Unzip the dataset, locate `data.yaml` and count the images in each split.

In [1]:
import os, zipfile, glob, yaml, shutil, random

# Configuration
ZIP_NAME   = 'player.yolov8.zip'          # ZIP exported from Roboflow (adjust if different)
extract_to = '/content/dataset_personas'

# Clean the destination folder so the cell can be re-run safely
if os.path.exists(extract_to):
    shutil.rmtree(extract_to)
os.makedirs(extract_to, exist_ok=True)

# Unzip the dataset
with zipfile.ZipFile(f'/content/{ZIP_NAME}', 'r') as z:
    z.extractall(extract_to)

# Locate data.yaml and show its content
yaml_files = glob.glob(f'{extract_to}/**/data.yaml', recursive=True)
DATA_YAML_PERSONS = yaml_files[0]
print(f'data.yaml: {DATA_YAML_PERSONS}')
print('--- content ---')
print(open(DATA_YAML_PERSONS).read())

# Count images per split
all_train = glob.glob(f'{extract_to}/**/train/images/*.jpg', recursive=True) + \
            glob.glob(f'{extract_to}/**/train/images/*.png', recursive=True)
all_valid = glob.glob(f'{extract_to}/**/valid/images/*.jpg', recursive=True) + \
            glob.glob(f'{extract_to}/**/valid/images/*.png', recursive=True)
all_test  = glob.glob(f'{extract_to}/**/test/images/*.jpg', recursive=True) + \
            glob.glob(f'{extract_to}/**/test/images/*.png', recursive=True)

print(f'\nTrain: {len(all_train)} imgs')
print(f'Valid: {len(all_valid)} imgs')
print(f'Test:  {len(all_test)} imgs')

data.yaml: /content/dataset_personas/data.yaml
--- content ---
train: ../train/images
val: ../valid/images
test: ../test/images

nc: 1
names: ['player']

roboflow:
  workspace: carloss-workspace-u4ffn
  project: carloss-workspace-u4ffn
  version: dataset
  license: Private
  url: https://app.roboflow.com/carloss-workspace-u4ffn/carloss-workspace-u4ffn/dataset

Train: 50 imgs
Valid: 0 imgs
Test:  0 imgs


## 2. Train/val split and configuration

If there is no validation set, build an 80/20 split and rewrite `data.yaml` with absolute paths.

In [2]:
# Dataset base folders
base = os.path.dirname(DATA_YAML_PERSONS)
train_imgs_dir = f'{base}/train/images'
train_lbls_dir = f'{base}/train/labels'
valid_imgs_dir = f'{base}/valid/images'
valid_lbls_dir = f'{base}/valid/labels'

# If there is no validation set (or it is empty), build an 80/20 split from train
if not os.path.exists(valid_imgs_dir) or len(os.listdir(valid_imgs_dir)) == 0:
    os.makedirs(valid_imgs_dir, exist_ok=True)
    os.makedirs(valid_lbls_dir, exist_ok=True)

    all_imgs = sorted(glob.glob(f'{train_imgs_dir}/*.jpg') +
                      glob.glob(f'{train_imgs_dir}/*.png'))
    print(f'Images in train: {len(all_imgs)}')

    random.seed(42)
    n_valid = max(2, int(len(all_imgs) * 0.20))
    valid_picks = random.sample(all_imgs, n_valid)

    # Move each selected image together with its label
    for img_path in valid_picks:
        name = os.path.splitext(os.path.basename(img_path))[0]
        ext  = os.path.splitext(img_path)[1]
        shutil.move(img_path, f'{valid_imgs_dir}/{name}{ext}')
        lbl_path = f'{train_lbls_dir}/{name}.txt'
        if os.path.exists(lbl_path):
            shutil.move(lbl_path, f'{valid_lbls_dir}/{name}.txt')

    print(f'Moved {n_valid} imgs from train to valid')

# Rewrite data.yaml with absolute paths
with open(DATA_YAML_PERSONS) as f:
    cfg = yaml.safe_load(f)

cfg['train'] = train_imgs_dir
cfg['val']   = valid_imgs_dir
if os.path.exists(f'{base}/test/images'):
    cfg['test'] = f'{base}/test/images'

with open(DATA_YAML_PERSONS, 'w') as f:
    yaml.dump(cfg, f)

print('\n--- data.yaml updated ---')
print(open(DATA_YAML_PERSONS).read())

# Final summary
print('\nFinal summary:')
print(f'  Train: {len(glob.glob(f"{train_imgs_dir}/*"))} imgs')
print(f'  Valid: {len(glob.glob(f"{valid_imgs_dir}/*"))} imgs')

Images in train: 50
Moved 10 imgs from train to valid

--- data.yaml updated ---
names:
- player
nc: 1
roboflow:
  license: Private
  project: carloss-workspace-u4ffn
  url: https://app.roboflow.com/carloss-workspace-u4ffn/carloss-workspace-u4ffn/dataset
  version: dataset
  workspace: carloss-workspace-u4ffn
test: ../test/images
train: /content/dataset_personas/train/images
val: /content/dataset_personas/valid/images


Final summary:
  Train: 40 imgs
  Valid: 10 imgs


## 3. Offline data augmentation (train only)

Generate transformed copies of the training images (with their bounding boxes) using Albumentations to expand a small dataset. The validation set is left untouched so the metrics stay honest.

In [3]:
!pip install -q albumentations
import albumentations as A
import cv2, os, glob

# Offline augmentation: only TRAIN is augmented; valid keeps real images
img_dir = train_imgs_dir
lbl_dir = train_lbls_dir
N_AUG   = 3                    # new copies generated per original image

transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.HueSaturationValue(p=0.4),
    A.ShiftScaleRotate(shift_limit=0.06, scale_limit=0.1, rotate_limit=12,
                       p=0.7, border_mode=cv2.BORDER_CONSTANT),
    A.Blur(blur_limit=3, p=0.1),
], bbox_params=A.BboxParams(format='yolo', label_fields=['classes'], min_visibility=0.3))

# Skip already-augmented images so the cell can be re-run without compounding
imgs = [p for p in glob.glob(f'{img_dir}/*.jpg') + glob.glob(f'{img_dir}/*.png')
        if '_aug' not in p]
created = 0

for img_path in imgs:
    name = os.path.splitext(os.path.basename(img_path))[0]
    lbl_path = f'{lbl_dir}/{name}.txt'
    if not os.path.exists(lbl_path):
        continue

    image = cv2.imread(img_path)
    bboxes, classes = [], []
    for line in open(lbl_path):
        parts = line.split()
        if len(parts) == 5:
            classes.append(int(parts[0]))
            bboxes.append([float(x) for x in parts[1:]])

    for i in range(N_AUG):
        try:
            aug = transform(image=image, bboxes=bboxes, classes=classes)
        except Exception:
            continue
        if len(aug['bboxes']) == 0:      # discard if all boxes were lost
            continue
        cv2.imwrite(f'{img_dir}/{name}_aug{i}.jpg', aug['image'])
        with open(f'{lbl_dir}/{name}_aug{i}.txt', 'w') as f:
            for c, b in zip(aug['classes'], aug['bboxes']):
                f.write(f'{c} {b[0]:.6f} {b[1]:.6f} {b[2]:.6f} {b[3]:.6f}\n')
        created += 1

total = len(glob.glob(f'{img_dir}/*.jpg')) + len(glob.glob(f'{img_dir}/*.png'))
print(f'Created {created} augmented images. Train now: {total} imgs')

/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


Created 105 augmented images. Train now: 145 imgs


## 4. Model training

Train YOLOv8s with on-the-fly augmentation and early stopping.

In [4]:
!pip install -q ultralytics

from ultralytics import YOLO
import os, glob

# Base model: pretrained YOLOv8s
model_persons = YOLO('yolov8s.pt')

# Training with on-the-fly data augmentation
results = model_persons.train(
    data=DATA_YAML_PERSONS,
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    project='runs/detect',
    name='volleyball_persons_custom',
    patience=20,
    save=True,
    verbose=True,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=10, translate=0.1, scale=0.5,
    fliplr=0.5, mosaic=1.0, mixup=0.15,
)

# Locate the best checkpoint produced
all_best = glob.glob('/content/**/volleyball_persons_custom*/weights/best.pt', recursive=True)
best_persons = max(all_best, key=os.path.getmtime)
print(f'\nBest model: {best_persons}')
print(f'Size: {os.path.getsize(best_persons)/(1024*1024):.1f} MB')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 79.7 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset_personas/data.yaml, degrees=10, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, fo

## 5. Export and download the model

Copy the best checkpoint (`best.pt`) and download it.

In [5]:
from google.colab import files
import shutil, os

SRC = '/content/runs/detect/runs/detect/volleyball_persons_custom/weights/best.pt'
DST = '/content/volleyball_persons_best.pt'

# Copy the model to a clean path and download it
if os.path.abspath(SRC) != os.path.abspath(DST):
    shutil.copy(SRC, DST)
    print(f'Copied to {DST}')

files.download(DST)
print(f'Downloading... size: {os.path.getsize(DST)/(1024*1024):.1f} MB')

Copied to /content/volleyball_persons_best.pt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloading... size: 21.5 MB
